# Detector de rostros ESCOM — CNN en Keras (Google Colab)

Reconoce a las 29 personas del grupo **a partir de su cara**. Es la versión para **Colab** del notebook de `kaggle/`. Flujo:

1. Baja las fotos de Hugging Face **persona por persona**, detecta la cara (MediaPipe) y guarda solo el recorte.
2. Divide en train / val / test **por sets** (una sesión de fotos nunca queda repartida).
3. Entrena una CNN desde cero y/o una CNN preentrenada con fine-tuning (TensorFlow / Keras).
4. Guarda métricas en el mismo formato que ArcFace y la CNN de PyTorch.

**Antes de correr:**
- *Entorno de ejecución → Cambiar tipo de entorno de ejecución → **GPU (T4)***.
- Icono de la llave 🔑 (panel izquierdo, *Secretos*): crea `HF_TOKEN` con un token de Hugging Face con **lectura** al dataset privado y activa *Acceso desde el notebook*.
- Colab borra todo al cerrar la sesión, por eso las caras recortadas y el modelo se guardan en **tu Google Drive** (la celda de abajo lo monta).

> ⚠️ **Datos biométricos.** Usa tu Drive personal, no compartas la carpeta `escom_caras`, y no hagas público el notebook con las salidas. Si lo subes al repo, limpia antes las salidas (*Editar → Borrar todas las salidas*).

In [ ]:
import os
from google.colab import drive
drive.mount("/content/drive")                       # aqui se guardan las caras recortadas y el modelo (privado, tu Drive)
GUARDAR = "/content/drive/MyDrive/escom_caras"      # carpeta en tu Drive
CARAS = f"{GUARDAR}/caras"
MANIFIESTO = f"{GUARDAR}/caras_manifest.csv"
os.makedirs(GUARDAR, exist_ok=True)

os.chdir("/content")
if not os.path.exists("repo"):
    !git clone --depth 1 https://github.com/GordoAbusivo/Detector-de-rostros repo
else:
    !git -C repo pull
os.chdir("/content/repo")

!pip install -q mediapipe huggingface_hub
!python -c "import tensorflow as tf, keras, mediapipe; print('TF', tf.__version__, '| Keras', keras.__version__, '| GPUs:', len(tf.config.list_physical_devices('GPU')))"

Si la última línea falla al importar `mediapipe` o `tensorflow` (conflicto de versiones), no pasa nada: el detector cae a **Haar de OpenCV**
(usa `--detector haar` en el paso 1). Si no dice `GPUs: 1`, activa la GPU en el tipo de entorno de ejecución y reinicia.

In [ ]:
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")   # el token nunca se escribe en el notebook
os.makedirs("/content/tmp", exist_ok=True)

## 1. Recortar las caras

`--detector auto` usa **MediaPipe** (pensado para caras pequeñas o lejanas; las fotos son de cuerpo entero) y, si no encuentra nada, **Haar** de OpenCV.
Las caras se guardan en tu Drive, así que es **reanudable**: si Colab se desconecta, monta Drive y vuelve a correr esta celda; salta a las personas ya terminadas.

Para una prueba rápida (≈1 min) agrega `--personas garcia_juan perez_juan --max-por-persona 60` y usa otra `--salida`.

In [ ]:
!python src/kaggle_preparar_caras.py --salida {CARAS} --tmp /content/tmp --tam 224 --margen 0.5 --detector auto

### Revisa los recortes antes de entrenar
Un recorte equivocado (otra persona del fondo, una cara falsa) le enseña a la red a confundirse. Mira una muestra y cuántas fotos se omitieron por persona.

In [ ]:
import glob, pandas as pd, matplotlib.pyplot as plt
from PIL import Image
m = pd.read_csv(MANIFIESTO)
print(m.groupby("persona").size().describe().round(0).to_dict(), "| caras por detector:", m["detector"].value_counts().to_dict())
print("fotos con mas de una cara (se tomo la mas grande):", int((m["n_caras"] > 1).sum()))

muestra = m.sample(24, random_state=0)
fig, ejes = plt.subplots(3, 8, figsize=(20, 8))
for e, f in zip(ejes.ravel(), muestra.itertuples()):
    e.imshow(Image.open(f"{CARAS}/{f.ruta}")); e.set_title(f.persona[:14], fontsize=8); e.axis("off")
plt.tight_layout(); plt.show()

omit = pd.concat(pd.read_csv(p) for p in glob.glob(f"{CARAS}/_manifest/*_omitidas.csv"))
print(omit["motivo"].str.slice(0, 22).value_counts().to_dict())
print(omit.groupby("persona").size().sort_values(ascending=False).head(8).to_dict(), "<- personas con mas fotos omitidas")

### Copia las caras al disco local de Colab
Leer miles de archivos pequeños directo desde Drive es lento; en el disco local de Colab el entrenamiento va mucho más rápido.

In [ ]:
!mkdir -p /content/caras_local && cp -r {CARAS}/. /content/caras_local/
CARAS_LOCAL = "/content/caras_local"

## 2. Entrenar

Los dos comandos reutilizan los mismos recortes y la misma división por sets. Las épocas son un máximo: `EarlyStopping` detiene cuando `val_loss` deja de mejorar y se conserva el mejor modelo.

**a) CNN desde cero** (línea base, ≈1 M de parámetros, entrada 128×128):

In [ ]:
!python src/entrenar_cnn_keras.py --modo scratch --tam 128 --epocas 60 --batch 64 --caras {CARAS_LOCAL} --manifiesto {MANIFIESTO}

**b) CNN preentrenada con fine-tuning** (recomendada). Primero se entrena solo la cabeza nueva (`--congelar 3` épocas) y luego toda la red.
Otras opciones: `--arq resnet50v2` o `--arq efficientnetb0`. Si se acaba la RAM, agrega `--sin-cache`.

In [ ]:
!python src/entrenar_cnn_keras.py --modo finetune --arq mobilenetv2 --tam 224 --epocas 25 --batch 64 --caras {CARAS_LOCAL} --manifiesto {MANIFIESTO}

### Guarda el modelo en tu Drive (¡antes de cerrar Colab!)
Colab borra `/content` al terminar la sesión. Esta celda copia los modelos y las métricas agregadas a tu Drive.

In [ ]:
!mkdir -p {GUARDAR}/models {GUARDAR}/resultados && cp -r models/. {GUARDAR}/models/ && cp -r resultados/. {GUARDAR}/resultados/
!ls {GUARDAR}/models

## 3. Resultados

In [ ]:
import matplotlib.image as mpimg
for nombre in ["keras_scratch", "keras_mobilenetv2"]:
    ruta = f"resultados/{nombre}/curvas.png"
    if os.path.exists(ruta):
        plt.figure(figsize=(12, 4)); plt.imshow(mpimg.imread(ruta)); plt.axis("off"); plt.title(nombre); plt.show()

!python src/comparar_modelos.py

In [ ]:
# Personas que mas se confunden (solo si quieres revisar los errores del mejor modelo)
nombre = "keras_mobilenetv2"
por_clase = pd.read_csv(f"resultados/{nombre}/por_clase.csv", index_col=0)
print(por_clase.sort_values("f1-score").head(8)[["precision", "recall", "f1-score", "support"]].round(3))
err = pd.read_csv(f"resultados/{nombre}/errores.csv")
print(err.groupby(["real", "predicho"]).size().sort_values(ascending=False).head(10))

## 4. Probar con una foto nueva

Sube una o varias fotos desde tu computadora; el script detecta la cara, la recorta igual que en el entrenamiento y muestra las 3 personas más probables.
Siempre responde una de las personas del grupo (no existe «desconocido»): una confianza baja significa que la red duda (cubrebocas, lentes, disfraz…).

In [ ]:
from google.colab import files
os.makedirs("fotos_prueba", exist_ok=True)
for nombre, datos in files.upload().items():
    open(f"fotos_prueba/{nombre}", "wb").write(datos)

!python src/predecir_keras.py fotos_prueba/* --modelo models/keras_mobilenetv2.keras --panel paneles

In [ ]:
import glob, matplotlib.image as mpimg
for p in sorted(glob.glob("paneles/*.png"))[:6]:
    plt.figure(figsize=(15, 5)); plt.imshow(mpimg.imread(p)); plt.axis("off"); plt.show()

## 5. Probar en vivo con la webcam (en tu laptop)

El script de webcam se corre en tu computadora, no en Colab:

1. Descarga de tu Drive (`escom_caras/models/`) `keras_mobilenetv2.keras` y `keras_mobilenetv2_clases.json`.
2. En tu laptop, clona el repo y pon esos dos archivos dentro de su carpeta `models/`.
3. Instala las dependencias (con la versión de OpenCV que abre ventanas) y ejecuta:

```bash
pip install -r requirements-keras.txt
pip uninstall -y opencv-python-headless && pip install opencv-python
python src/webcam_keras.py --modelo models/keras_mobilenetv2.keras              # webcam de la laptop
python src/webcam_keras.py --modelo models/keras_mobilenetv2.keras --camara 1   # webcam externa
python src/webcam_keras.py --modelo models/keras_mobilenetv2.keras --video prueba.mp4   # o un video grabado
```

Teclas: `q` o `Esc` salen, `s` guarda una captura. El recuadro es **verde** con confianza ≥ 70 %, **naranja** ≥ 40 % y **rojo** por debajo.
El modelo es biométrico: no lo subas a GitHub ni lo compartas fuera del equipo.

## 6. Qué puede ir al repo

Solo números agregados: `resultados/*/metricas.json`, `historial.csv`, `curvas.png` y `resultados/comparacion.md`. Nunca las caras, el modelo ni vectores.